In [9]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Exact location of your ZIP file
zip_path = Path('/content/drive/MyDrive/archive (1).zip')

print("File found:", zip_path.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File found: True


In [10]:

import zipfile

# Open the ZIP without extracting it
with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

    # Count patient files in each hospital
    hospital_A = [
        f for f in files
        if f.startswith("training_setA/") and f.endswith(".psv")
    ]

    hospital_B = [
        f for f in files
        if f.startswith("training_setB/") and f.endswith(".psv")
    ]

    print("Hospital A:", len(hospital_A), "patients")
    print("Hospital B:", len(hospital_B), "patients")


Hospital A: 20336 patients
Hospital B: 20000 patients


In [11]:

from pathlib import Path
import zipfile

# Where to extract Hospital A
extract_folder = Path("/content/sepsis_dataset")

# Extract Hospital A from the ZIP
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(
        path=extract_folder,
        members=hospital_A
    )

# Find patient files, including inside subfolders
patient_files = sorted(
    (extract_folder / "training_setA").rglob("*.psv")
)

# Find the actual folder containing patient files
print("Extracted patients:", len(patient_files))

if patient_files:
    data_folder = patient_files[0].parent
    print("Patient folder:", data_folder)
    print("First patient:", patient_files[0].name)


Extracted patients: 20336
Patient folder: /content/sepsis_dataset/training_setA/training
First patient: p000001.psv


In [12]:

import pandas as pd

# Open the first patient's file
patient = pd.read_csv(patient_files[0], sep="|")

# Check its dimensions
print("Patient:", patient_files[0].name)
print("Rows:", patient.shape[0])
print("Columns:", patient.shape[1])

# Display the first 5 hourly observations
display(patient.head())

# Display all column names
print("\nColumn names:")
print(patient.columns.tolist())


Patient: p000001.psv
Rows: 54
Columns: 41


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0



Column names:
['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel']


In [13]:

import pandas as pd

all_patients = []

for i, file in enumerate(patient_files):

    # Read one patient's data
    df = pd.read_csv(file, sep="|")

    # Add patient identifier
    df["PatientID"] = file.stem

    # Add hour index (0, 1, 2, ...)
    df["Hour"] = range(len(df))

    # Store the patient's data
    all_patients.append(df)

    if (i + 1) % 2000 == 0:
        print(f"Loaded {i + 1} patients")

# Combine all patients into one table
data_A = pd.concat(all_patients, ignore_index=True)

# Free the temporary list
del all_patients

print("\nTotal patients:", data_A["PatientID"].nunique())
print("Total hourly observations:", len(data_A))
print("Total columns:", data_A.shape[1])

display(data_A.head())


Loaded 2000 patients
Loaded 4000 patients
Loaded 6000 patients
Loaded 8000 patients
Loaded 10000 patients
Loaded 12000 patients
Loaded 14000 patients
Loaded 16000 patients
Loaded 18000 patients
Loaded 20000 patients

Total patients: 20336
Total hourly observations: 790215
Total columns: 43


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,PatientID,Hour
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001,0
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001,1
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001,2
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001,3
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001,4


In [14]:

from sklearn.model_selection import train_test_split

# Determine whether each patient ever had sepsis
patient_status = data_A.groupby("PatientID")["SepsisLabel"].max()

# Split patient IDs (80% training, 20% validation)
train_ids, val_ids = train_test_split(
    patient_status.index,
    test_size=0.2,
    random_state=42,
    stratify=patient_status.values
)

# Select all hourly rows belonging to each group
train_A = data_A[data_A["PatientID"].isin(train_ids)].copy()
val_A = data_A[data_A["PatientID"].isin(val_ids)].copy()

# Check that no patient appears in both groups
assert set(train_ids).isdisjoint(set(val_ids))

# Display results
print("TRAINING SET")
print("Patients:", train_A["PatientID"].nunique())
print("Hourly observations:", len(train_A))
print("Septic patients:",
      int(train_A.groupby("PatientID")["SepsisLabel"].max().sum()))

print("\nVALIDATION SET")
print("Patients:", val_A["PatientID"].nunique())
print("Hourly observations:", len(val_A))
print("Septic patients:",
      int(val_A.groupby("PatientID")["SepsisLabel"].max().sum()))


TRAINING SET
Patients: 16268
Hourly observations: 631120
Septic patients: 1432

VALIDATION SET
Patients: 4068
Hourly observations: 159095
Septic patients: 358


In [15]:

# Identify the 40 original input features
feature_cols = [
    col for col in train_A.columns
    if col not in ["SepsisLabel", "PatientID", "Hour"]
]

# Calculate missing percentage in training data only
missing_pct = train_A[feature_cols].isna().mean() * 100

# Sort from most missing to least missing
missing_pct = missing_pct.sort_values(ascending=False)

print("Number of original features:", len(feature_cols))

print("\nFeatures with at least 99% missing values:")
print(missing_pct[missing_pct >= 99].round(2))

print("\nFeatures with 95% to 99% missing values:")
print(missing_pct[(missing_pct >= 95) & (missing_pct < 99)].round(2))

# Features with no variation in observed values
constant_features = [
    col for col in feature_cols
    if train_A[col].nunique(dropna=True) <= 1
]

print("\nFeatures with no usable variation:")
print(constant_features)


Number of original features: 40

Features with at least 99% missing values:
EtCO2               100.00
TroponinI            99.88
Bilirubin_direct     99.86
Fibrinogen           99.24
dtype: float64

Features with 95% to 99% missing values:
Bilirubin_total    98.78
Alkalinephos       98.53
AST                98.49
Lactate            96.52
PTT                95.16
SaO2               95.04
Calcium            95.01
dtype: float64

Features with no usable variation:
['EtCO2']


In [16]:

# Select features with at least 99% missing values
drop_cols = missing_pct[missing_pct >= 99].index.tolist()

# Apply the SAME feature removal to train and validation
train_base = train_A.drop(columns=drop_cols).copy()
val_base = val_A.drop(columns=drop_cols).copy()

# List of remaining model input features
keep_cols = [col for col in feature_cols if col not in drop_cols]

print("Removed features:", drop_cols)
print("Remaining model features:", len(keep_cols))

print("\nTraining shape:", train_base.shape)
print("Validation shape:", val_base.shape)

# Check that both sets have identical columns
assert train_base.columns.tolist() == val_base.columns.tolist()


Removed features: ['EtCO2', 'TroponinI', 'Bilirubin_direct', 'Fibrinogen']
Remaining model features: 36

Training shape: (631120, 39)
Validation shape: (159095, 39)


In [17]:

# Sort observations chronologically within each patient
train_filled = train_base.sort_values(["PatientID", "Hour"]).copy()
val_filled = val_base.sort_values(["PatientID", "Hour"]).copy()

# Treat missing ICU unit information as "unknown"
unit_cols = ["Unit1", "Unit2"]
numeric_cols = [c for c in keep_cols if c not in unit_cols]

# 1. Forward fill within each patient
train_filled[numeric_cols] = (
    train_filled.groupby("PatientID")[numeric_cols].ffill()
)

val_filled[numeric_cols] = (
    val_filled.groupby("PatientID")[numeric_cols].ffill()
)

# 2. Calculate medians from TRAINING data only
train_medians = train_filled[numeric_cols].median()

# 3. Fill remaining missing values using training medians
train_filled[numeric_cols] = (
    train_filled[numeric_cols].fillna(train_medians)
)

val_filled[numeric_cols] = (
    val_filled[numeric_cols].fillna(train_medians)
)

# 4. Encode missing ICU unit information as unknown (-1)
train_filled[unit_cols] = train_filled[unit_cols].fillna(-1)
val_filled[unit_cols] = val_filled[unit_cols].fillna(-1)

# Verify that all model inputs are complete
print("Training missing values:",
      train_filled[keep_cols].isna().sum().sum())

print("Validation missing values:",
      val_filled[keep_cols].isna().sum().sum())


Training missing values: 0
Validation missing values: 0


In [18]:

from sklearn.preprocessing import StandardScaler

# Make sure Step 4 successfully handled missing values
assert train_filled[keep_cols].isna().sum().sum() == 0
assert val_filled[keep_cols].isna().sum().sum() == 0

# Categorical/binary features don't need standardization
categorical_cols = ["Gender", "Unit1", "Unit2"]

# Select continuous features
scale_cols = [
    col for col in keep_cols
    if col not in categorical_cols
]

# Separate inputs (X) from target labels (y)
X_train = train_filled[keep_cols].copy()
X_val = val_filled[keep_cols].copy()

y_train = train_filled["SepsisLabel"].copy()
y_val = val_filled["SepsisLabel"].copy()

# Create the standardizer
scaler = StandardScaler()

# Learn mean and std from TRAINING data only
X_train[scale_cols] = scaler.fit_transform(
    X_train[scale_cols]
)

# Apply the same transformation to validation
X_val[scale_cols] = scaler.transform(
    X_val[scale_cols]
)

# Check results
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("\nHR after standardization:")
print("Training mean:", round(X_train["HR"].mean(), 4))
print("Training std:", round(X_train["HR"].std(ddof=0), 4))

print("\nMissing values:")
print("Training:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())


Training shape: (631120, 36)
Validation shape: (159095, 36)

HR after standardization:
Training mean: -0.0
Training std: 1.0

Missing values:
Training: 0
Validation: 0


In [19]:

from sklearn.linear_model import LogisticRegression

# Create our baseline Logistic Regression model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model on Hospital A's training data
model.fit(X_train, y_train)

print("Logistic Regression training complete!")
print("Number of features:", model.n_features_in_)
print("Number of iterations:", model.n_iter_[0])


Logistic Regression training complete!
Number of features: 36
Number of iterations: 18


In [20]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Predict probability of sepsis for each validation hour
y_prob = model.predict_proba(X_val)[:, 1]

# Convert probabilities into predictions using threshold 0.5
y_pred = (y_prob >= 0.5).astype(int)

# Calculate evaluation metrics
print("===== BASELINE LOGISTIC REGRESSION =====")

print("AUROC:", round(roc_auc_score(y_val, y_prob), 4))
print("AUPRC:", round(average_precision_score(y_val, y_prob), 4))
print("Precision:", round(precision_score(y_val, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_val, y_pred, zero_division=0), 4))

print("\nActual positive hours:", int(y_val.sum()))
print("Predicted positive hours:", int(y_pred.sum()))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))


===== BASELINE LOGISTIC REGRESSION =====
AUROC: 0.759
AUPRC: 0.0803
Precision: 0.1005
Recall: 0.0061
F1-score: 0.0116

Actual positive hours: 3426
Predicted positive hours: 209

Confusion Matrix:
[[155481    188]
 [  3405     21]]


In [21]:

# Compare different probability thresholds
thresholds = [0.50, 0.20, 0.10, 0.05, 0.02]

results = []

for threshold in thresholds:
    predictions = (y_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "Positive predictions": predictions.sum(),
        "False positive hours": (
            (predictions == 1) & (y_val.to_numpy() == 0)
        ).sum()
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))


,Threshold,Precision,Recall,F1,Positive predictions,False positive hours
0,0.50,0.1005,0.0061,0.0116,209,188
1,0.20,0.1430,0.0429,0.0660,1028,881
2,0.10,0.1359,0.1124,0.1230,2832,2447
3,0.05,0.1113,0.2820,0.1596,8677,7711
4,0.02,0.0470,0.7026,0.0880,51258,48851


Conclusion:
The baseline logistic regression model achieved an AUROC of 0.759 and an AUPRC of 0.0803. At the default threshold of 0.5, recall was extremely low (0.61%). Lower thresholds improved recall considerably but increased false positives. In the future experiments, we will investigate class weighting, clinical feature groups, and missingness indicators.